# Individual/longitudinal plots

In [135]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr


In [136]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [137]:
# biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom'
biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed.biom'

In [138]:
feature_df = biom_to_df(biom_path)
feature_df

,LAMI.RD001.D0.C1,LAMI.RD001.D14.C1,LAMI.RD004.D0.C2,LAMI.RD001.D0.C2,LAMI.RD004.D28.C1,LAMI.RD011.D0.C2,LAMI.RD001.D14.C2,LAMI.RD004.D28.C2,LAMI.RD017.D14.C2,LAMI.RD011.D14.C1,...,LAMI.RD319.D16.C1,LAMI.RD318.D16.C1,LAMI.RD319.D25.C2,LAMI.RD318.D18.C1,LAMI.RD318.D4.C2,LAMI.RD319.D16.C2,LAMI.RD319.D28.C1,LAMI.RD318.D9.C2,LAMI.RD319.D28.C2,LAMI.RD319.D2.C1
g__Staphylococcus,0.084107,0.120456,0.438313,0.092598,0.088087,0.565667,0.117803,0.057575,0.465906,0.547360,...,0.015389,0.019103,0.018042,0.022022,0.107456,0.037676,0.011144,0.036880,0.013797,0.031839
g__uncultured,0.016185,0.007694,0.009552,0.012205,0.004245,0.012205,0.010613,0.008490,0.008490,0.021756,...,0.945078,0.874768,0.954895,0.873176,0.688512,0.905545,0.935261,0.912974,0.962324,0.931016
g__Lawsonella,0.034757,0.120987,0.025206,0.057575,0.014327,0.077209,0.054656,0.004776,0.168745,0.265057,...,0.009286,0.011940,0.014327,0.005837,0.027594,0.015123,0.025471,0.006102,0.014327,0.011940
g__Corynebacterium,0.072964,0.242770,0.224993,0.135049,0.255771,0.058371,0.144866,0.193951,0.034757,0.027063,...,0.004245,0.011144,0.005306,0.006633,0.019369,0.012735,0.010082,0.007164,0.003980,0.007164
g__Streptococcus,0.109843,0.065535,0.043513,0.097639,0.111966,0.040594,0.121252,0.189175,0.008756,0.022552,...,0.002919,0.019103,0.000265,0.006368,0.006633,0.000531,0.001857,0.002919,0.000265,0.000531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Hymenobacter,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
g__Bacteroides,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
g__Absconditabacteriales_(SR1),0.010082,0.000000,0.000000,0.005041,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
g__Sphingomonas,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [139]:
# Let's assume feature_df_T is your sample-by-feature table (after dropping SampleID)
# Step 1: Add pseudocount to avoid log(0)
pseudocount = 1e-6
clr_input = feature_df + pseudocount

# Step 2: Take log
log_data = np.log(clr_input)

# Step 3: Center the log values (CLR)
clr_transformed = log_data.subtract(log_data.mean(axis=1), axis=0)
clr_transformed

,LAMI.RD001.D0.C1,LAMI.RD001.D14.C1,LAMI.RD004.D0.C2,LAMI.RD001.D0.C2,LAMI.RD004.D28.C1,LAMI.RD011.D0.C2,LAMI.RD001.D14.C2,LAMI.RD004.D28.C2,LAMI.RD017.D14.C2,LAMI.RD011.D14.C1,...,LAMI.RD319.D16.C1,LAMI.RD318.D16.C1,LAMI.RD319.D25.C2,LAMI.RD318.D18.C1,LAMI.RD318.D4.C2,LAMI.RD319.D16.C2,LAMI.RD319.D28.C1,LAMI.RD318.D9.C2,LAMI.RD319.D28.C2,LAMI.RD319.D2.C1
g__Staphylococcus,-0.829930,-0.470738,0.820901,-0.733761,-0.783697,1.075974,-0.493010,-1.208929,0.881952,1.043075,...,-2.528335,-2.312125,-2.369280,-2.169957,-0.584947,-1.632990,-2.851084,-1.654342,-2.637527,-1.801320
g__uncultured,-1.513734,-2.257243,-2.041046,-1.795946,-2.851845,-1.795946,-1.935696,-2.158816,-2.158816,-1.217904,...,2.553409,2.476100,2.563743,2.474279,2.236674,2.510679,2.542968,2.518849,2.571493,2.538418
g__Lawsonella,0.170953,1.418228,-0.150357,0.675641,-0.715219,0.969063,0.623621,-1.813692,1.750931,2.202485,...,-1.148818,-0.897527,-0.715219,-1.613059,-0.059846,-0.661156,-0.139886,-1.568615,-0.715219,-0.897527
g__Corynebacterium,0.565306,1.767449,1.691406,1.180976,1.819616,0.342165,1.251147,1.542939,-0.176253,-0.426469,...,-2.278655,-1.313720,-2.055559,-1.832453,-0.760968,-1.180200,-1.413794,-1.755503,-2.343178,-1.755503
g__Streptococcus,1.923170,1.406698,0.997184,1.805388,1.942309,0.927757,2.021986,2.466781,-0.606084,0.339990,...,-1.704468,0.174013,-4.098943,-0.924495,-0.883679,-3.407675,-2.156257,-1.704468,-4.098943,-3.407675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Hymenobacter,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,...,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776,-0.084776
g__Bacteroides,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,...,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471,-0.288471
g__Absconditabacteriales_(SR1),9.050085,-0.168546,-0.168546,8.357037,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,...,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546,-0.168546
g__Sphingomonas,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,...,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957,-0.265957


In [140]:
metadata_df = pd.read_csv('../Metadata/metadata_df.csv')
metadata_df['local_lesion_severity']
metadata_df

,Unnamed: 0,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,...,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group
0,0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,...,33.765638,acne,RD308,Acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
1,1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,...,31.919478,acne,RD310,Acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
2,2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,...,22.152503,acne,RD305,Healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL
3,3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,...,27.397918,acne,RD306,Acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L
4,4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,...,28.819451,acne,RD306,Acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,...,21.946648,acne,RD317,Acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L
262,262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,...,26.344240,control,RD001,Healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy
263,263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,...,16.359699,control,RD014,Healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy
264,264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,...,22.494605,acne,RD314,Acne,PP_314,PP_314C1,Lesional_C1,Acne_L,low,low Acne_L


In [146]:
metadata_df['local_lesion_severity'].value_counts()

local_lesion_severity
0    84
1    56
2    37
3    36
4    28
5    19
6     6
Name: count, dtype: int64

In [141]:
# Transpose so that rows = samples, columns = features
feature_df_T = clr_transformed.T
feature_df_T['SampleID'] = feature_df_T.index
feature_df_T

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Dietzia,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella,SampleID
LAMI.RD001.D0.C1,-0.829930,-1.513734,0.170953,0.565306,1.923170,1.940983,4.003059,2.139191,1.315980,6.537142,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,9.050085,-0.265957,-0.124317,LAMI.RD001.D0.C1
LAMI.RD001.D14.C1,-0.470738,-2.257243,1.418228,1.767449,1.406698,1.230478,2.963722,2.066623,3.369885,5.343515,...,-0.480306,-0.311016,-0.613708,7.295833,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD001.D14.C1
LAMI.RD004.D0.C2,0.820901,-2.041046,-0.150357,1.691406,0.997184,1.464079,1.172075,1.928479,3.182429,1.364758,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD004.D0.C2
LAMI.RD001.D0.C2,-0.733761,-1.795946,0.675641,1.180976,1.805388,1.606246,3.979628,1.851521,3.811706,6.274638,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,8.357037,-0.265957,-0.124317,LAMI.RD001.D0.C2
LAMI.RD004.D28.C1,-0.783697,-2.851845,-0.715219,1.819616,1.942309,2.974269,1.273845,2.304700,1.903655,1.364758,...,-0.480306,-0.311016,5.662267,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD004.D28.C1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D16.C2,-1.632990,2.510679,-0.661156,-1.180200,-3.407675,-1.712772,-7.741210,-1.770987,-6.973267,-5.603424,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D16.C2
LAMI.RD319.D28.C1,-2.851084,2.542968,-0.139886,-1.413794,-2.156257,-1.202448,0.037379,-8.046963,-6.973267,-5.603424,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D28.C1
LAMI.RD318.D9.C2,-1.654342,2.518849,-1.568615,-1.755503,-1.704468,0.742816,-1.465234,0.019204,-0.697291,-5.603424,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD318.D9.C2
LAMI.RD319.D28.C2,-2.637527,2.571493,-0.715219,-2.343178,-4.098943,-8.393585,-2.156502,-1.770987,-1.388559,-0.018716,...,-0.480306,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D28.C2


In [142]:
# Merge with metadata
merged = pd.merge(feature_df_T, metadata_df[['SampleID', 'local_lesion_severity']], on='SampleID')
merged

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella,SampleID,local_lesion_severity
0,-0.829930,-1.513734,0.170953,0.565306,1.923170,1.940983,4.003059,2.139191,1.315980,6.537142,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,9.050085,-0.265957,-0.124317,LAMI.RD001.D0.C1,0
1,-0.470738,-2.257243,1.418228,1.767449,1.406698,1.230478,2.963722,2.066623,3.369885,5.343515,...,-0.311016,-0.613708,7.295833,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD001.D14.C1,0
2,0.820901,-2.041046,-0.150357,1.691406,0.997184,1.464079,1.172075,1.928479,3.182429,1.364758,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD004.D0.C2,0
3,-0.733761,-1.795946,0.675641,1.180976,1.805388,1.606246,3.979628,1.851521,3.811706,6.274638,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,8.357037,-0.265957,-0.124317,LAMI.RD001.D0.C2,0
4,-0.783697,-2.851845,-0.715219,1.819616,1.942309,2.974269,1.273845,2.304700,1.903655,1.364758,...,-0.311016,5.662267,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD004.D28.C1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,-1.632990,2.510679,-0.661156,-1.180200,-3.407675,-1.712772,-7.741210,-1.770987,-6.973267,-5.603424,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D16.C2,5
213,-2.851084,2.542968,-0.139886,-1.413794,-2.156257,-1.202448,0.037379,-8.046963,-6.973267,-5.603424,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D28.C1,3
214,-1.654342,2.518849,-1.568615,-1.755503,-1.704468,0.742816,-1.465234,0.019204,-0.697291,-5.603424,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD318.D9.C2,3
215,-2.637527,2.571493,-0.715219,-2.343178,-4.098943,-8.393585,-2.156502,-1.770987,-1.388559,-0.018716,...,-0.311016,-0.613708,-0.365025,-0.084776,-0.288471,-0.168546,-0.265957,-0.124317,LAMI.RD319.D28.C2,4


In [143]:
# Drop non-numeric columns before averaging
numeric_cols = merged.select_dtypes(include='number').columns
# Exclude the grouping column itself (optional)
feature_cols = numeric_cols.drop('local_lesion_severity')

# Group by severity and compute the mean of only the features
grouped = merged.groupby('local_lesion_severity')[feature_cols].mean()
grouped

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Anaerostipes,g__Dietzia,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella
local_lesion_severity,,,,,,,,,,,,,,,,,,,,,
0,0.201126,-0.488105,0.456200,0.544081,0.463284,0.688594,0.875778,1.161151,-0.080425,0.914732,...,-0.030213,0.309158,0.293762,0.514195,0.210827,0.255897,-0.118249,0.160051,-0.162536,0.246213
1,-0.125380,0.626475,0.449857,-0.240612,-0.380627,-0.595432,-0.124502,-0.268137,-0.023759,-0.452693,...,-0.010241,0.096286,-0.311016,-0.237131,-0.253331,-0.084776,0.074262,-0.056852,-0.140437,-0.124317
2,0.343902,0.000107,0.375810,0.390213,0.374770,0.090162,-0.311220,-0.226505,0.410945,-0.319671,...,-0.381071,-0.121652,-0.311016,-0.403041,0.280785,-0.084776,-0.288471,-0.008983,0.084487,-0.124317
3,0.234469,-0.347705,-0.046330,-0.121982,0.287080,0.793670,0.489231,-0.350110,0.110903,0.334200,...,0.180792,-0.290124,-0.311016,-0.613708,0.238805,-0.084776,0.112867,0.063601,0.072510,-0.124317
4,-0.602123,-0.072519,-1.352702,-0.583222,-0.659024,-1.473980,-1.287565,-0.857233,0.532546,0.174994,...,0.366286,-0.173871,0.163412,0.219977,-0.365025,-0.084776,0.592545,-0.168546,-0.042568,-0.124317
5,-0.596993,0.543165,-1.380195,-0.739645,-0.829481,-0.290204,-0.988355,-0.717356,-1.809899,-2.100815,...,0.062217,-0.131640,0.965193,0.106183,-0.365025,-0.084776,-0.288471,-0.168546,0.231175,0.262804
6,0.716550,-0.729816,-0.045514,0.456159,1.414837,2.656976,1.830352,2.264644,3.376287,3.419402,...,-0.381071,-0.480306,-0.311016,5.519052,-0.365025,-0.084776,-0.288471,-0.168546,3.675997,-0.124317


In [144]:
# Variance across severity levels for each feature
feature_variance = grouped.var()

# Top N features with highest variance
top_variable_features = feature_variance.sort_values(ascending=False).head(10)
print(top_variable_features)


g__Chryseobacterium    6.657670
g__Aeromonas           6.280122
g__Treponema           4.609256
g__Porphyromonas       4.403455
g__Alloprevotella      3.864506
g__Aggregatibacter     3.414570
g__Brochothrix         3.296576
g__Neisseria           2.806234
g__Gemella             2.517566
g__Caulobacter         2.453378
dtype: float64


In [145]:
import matplotlib.pyplot as plt

grouped[list(top_variable_features.index)].plot(marker='o')
plt.xlabel('Lesion Severity')
plt.ylabel('Mean Abundance')
plt.title('Top Features Showing Most Variation Across Severity')
plt.legend(title='Feature', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('severity_patterns_V4.png', dpi=600)
